# Task 4 — General Health Query Chatbot

A conversational agent for **general health information** using prompt engineering, with **safety filtering** for high-risk topics.

**Not medical advice.** This demo is for learning only.

## 1. Dependencies and environment

Install packages (once per environment):

```bash
pip install openai huggingface_hub python-dotenv transformers accelerate torch safetensors
```

### API keys (do not commit secrets)

| Variable | Purpose |
|----------|--------|
| `OPENAI_API_KEY` | Required if using OpenAI |
| `OPENAI_MODEL` | Optional; default `gpt-4o-mini` |
| `HF_TOKEN` | Optional for public models; helps with gated models / rate limits |
| `HF_MODEL` | Optional; HF model id for chat |

You can create a `.env` file in the project root:

```
OPENAI_API_KEY=sk-...
```

The next cell loads `.env` automatically if `python-dotenv` is installed.

In [8]:
pip install openai huggingface_hub python-dotenv transformers accelerate torch safetensors

In [9]:
"""Load environment variables from .env when present."""
import os
from pathlib import Path

try:
    from dotenv import load_dotenv
    # Search upward from cwd for .env
    load_dotenv(Path.cwd() / ".env")
    load_dotenv(Path.cwd().parent / ".env")
except ImportError:
    pass  # python-dotenv optional; use exported vars in shell instead

print("Environment ready. For OpenAI set OPENAI_API_KEY. For HF local model, pip-install transformers (see Colab cell). HF_TOKEN optional for public models.")

Environment ready. For OpenAI set OPENAI_API_KEY. For HF local model, pip-install transformers (see Colab cell). HF_TOKEN optional for public models.


### Google Colab — Hugging Face token (recommended)

**Do not paste API keys into notebook cells** (they can leak if you share the notebook).

1. In Colab, click the **key icon** (Secrets) in the left sidebar.
2. Create a secret named exactly: **`HF_TOKEN`**
3. Paste your Hugging Face token as the value.
4. Turn on **Notebook access** for that secret.
5. Run the next cell so `HF_TOKEN` is loaded into `os.environ`.

Fine-grained tokens work if they include permission to use **Inference** / call the model you set in `HF_MODEL`.

In [10]:
"""Colab: install deps + load HF_TOKEN from Colab Secrets. No-op elsewhere."""
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openai",
            "huggingface_hub",
            "python-dotenv",
            "transformers",
            "accelerate",
            "torch",
            "safetensors",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("Colab: HF_TOKEN loaded from Secrets (name must be HF_TOKEN).")
    except Exception as exc:
        print(
            "Colab: could not read HF_TOKEN from Secrets. Add secret 'HF_TOKEN' and enable notebook access."
        )
        print("Details:", exc)
else:
    print("Not Colab: use .env or `export HF_TOKEN=...` in your terminal.")

Colab: HF_TOKEN loaded from Secrets (name must be HF_TOKEN).


## 2. System prompt and safety filter

- **System prompt**: defines the assistant persona.
- **`safety_filter`**: detects high-risk phrases and returns the **mandatory disclaimer** required for this task.

In [11]:
# ---------------------------------------------------------------------------
# Persona: instructs the model how to behave (prompt engineering)
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """Act as a helpful, friendly, and professional medical assistant.

Guidelines:
- Provide general health education in clear, accessible language.
- Be empathetic and respectful.
- Do not provide a definitive diagnosis or prescribe treatments; encourage seeing a qualified clinician when appropriate.
- If the user describes a possible emergency or severe symptoms, urge them to seek immediate professional or emergency care.
""".strip()

# Exact mandatory disclaimer when high-risk keywords are detected (task requirement)
MANDATORY_DISCLAIMER = (
    "I am an AI, not a doctor. If you are experiencing a medical emergency, "
    "please call your local emergency services immediately."
)

# High-risk terms: emergency, self-harm, severe acute symptoms, etc.
# Extend this list carefully; avoid overly generic words to reduce false positives.
HIGH_RISK_KEYWORDS = frozenset(
    {
        "emergency",
        "suicide",
        "kill myself",
        "end my life",
        "chest pain",
        "can't breathe",
        "cannot breathe",
        "heart attack",
        "stroke",
        "severe bleeding",
        "unconscious",
        "overdose",
        "poisoning",
    }
)


def safety_filter(user_text: str) -> tuple[bool, str]:
    """
    Scan user input for high-risk keywords.

    Returns:
        (triggered, disclaimer):
            - triggered True if any keyword matches (case-insensitive substring).
            - disclaimer is MANDATORY_DISCLAIMER when triggered, else empty string.
    """
    if not user_text or not user_text.strip():
        return False, ""

    lowered = user_text.lower()
    for phrase in HIGH_RISK_KEYWORDS:
        if phrase in lowered:
            return True, MANDATORY_DISCLAIMER
    return False, ""

## 3. LLM backends (OpenAI and Hugging Face)

Modular functions: one entry point `complete_chat` dispatches by `provider`.

In [12]:
from typing import Literal, Optional

Provider = Literal["openai", "huggingface"]


def _messages_to_openai_format(
    history: list[dict[str, str]],
) -> list[dict[str, str]]:
    """history items: {'role': 'user'|'assistant', 'content': '...'}"""
    out = [{"role": "system", "content": SYSTEM_PROMPT}]
    out.extend(history)
    return out


def complete_chat_openai(history: list[dict[str, str]]) -> str:
    """Call OpenAI Chat Completions API."""
    from openai import OpenAI

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "OPENAI_API_KEY is not set. Export it or add it to .env"
        )

    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    client = OpenAI(api_key=api_key)
    resp = client.chat.completions.create(
        model=model,
        messages=_messages_to_openai_format(history),
        temperature=0.4,
        max_tokens=1024,
    )
    choice = resp.choices[0]
    if not choice.message.content:
        return ""
    return choice.message.content.strip()


def _hf_smollm2_chat_prompt(messages: list[dict[str, str]]) -> str:
    """Build prompt using SmolLM2 / ChatML-style tokens (works with HF serverless text_generation)."""
    im_start = "<|im_start|>"
    im_end = "<|" + "im_end" + "|>"  # ChatML end-of-turn (SmolLM2-compatible)
    parts: list[str] = []
    for msg in messages:
        parts.append(f"{im_start}{msg['role']}\n{msg['content']}{im_end}\n")
    parts.append(f"{im_start}assistant\n")
    return "".join(parts)


# Hugging Face Hub cloud inference often fails with "model not supported" on the router.
# Running the model **locally** with `transformers` is reliable on Colab (GPU) or CPU (slower).
_HF_LOCAL_MODEL = None
_HF_LOCAL_TOKENIZER = None
_HF_LOCAL_MODEL_ID: Optional[str] = None


def _get_hf_local_model(model_id: str, hf_token: Optional[str]):
    """Lazy-load tokenizer + model once per (kernel, model_id)."""
    global _HF_LOCAL_MODEL, _HF_LOCAL_TOKENIZER, _HF_LOCAL_MODEL_ID
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if _HF_LOCAL_MODEL is not None and _HF_LOCAL_MODEL_ID == model_id:
        return _HF_LOCAL_MODEL, _HF_LOCAL_TOKENIZER

    load_kw: dict = {}
    if hf_token:
        load_kw["token"] = hf_token

    _HF_LOCAL_TOKENIZER = AutoTokenizer.from_pretrained(model_id, **load_kw)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    _HF_LOCAL_MODEL = AutoModelForCausalLM.from_pretrained(
        model_id,
        **load_kw,
        device_map="auto",
        torch_dtype=dtype,
    )
    _HF_LOCAL_MODEL_ID = model_id
    return _HF_LOCAL_MODEL, _HF_LOCAL_TOKENIZER


def complete_chat_huggingface(history: list[dict[str, str]]) -> str:
    """Run a small instruct model locally via `transformers` (avoids HF Inference API / router issues)."""
    import torch

    hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    model_id = os.getenv("HF_MODEL", "HuggingFaceTB/SmolLM2-360M-Instruct")

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history)
    prompt = _hf_smollm2_chat_prompt(messages)

    model, tokenizer = _get_hf_local_model(model_id, hf_token)
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.4,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def complete_chat(
    history: list[dict[str, str]], provider: Provider = "openai"
) -> str:
    """Dispatch to the selected LLM backend."""
    if provider == "openai":
        return complete_chat_openai(history)
    if provider == "huggingface":
        return complete_chat_huggingface(history)
    raise ValueError(f"Unknown provider: {provider}")

## 4. Chat loop

Interactive session: type questions such as *"What causes a sore throat?"* or *"Is paracetamol safe for children?"*

Commands: `quit` or `exit` to stop.

In [13]:
def run_health_chat_loop(provider: Provider = "openai") -> None:
    """
    Simple REPL: maintain conversation history and apply safety_filter on each user turn.
    When high-risk keywords are detected, print the mandatory disclaimer before the model reply.
    """
    history: list[dict[str, str]] = []

    print("Health assistant chat. Type 'quit' or 'exit' to stop.\n")
    print(f"Using provider: {provider}\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except EOFError:
            break

        if not user_input:
            continue
        if user_input.lower() in {"quit", "exit", "q"}:
            print("Goodbye.")
            break

        risky, disclaimer = safety_filter(user_input)
        if risky:
            print(f"\n[Safety] {disclaimer}\n")

        history.append({"role": "user", "content": user_input})

        try:
            reply = complete_chat(history, provider=provider)
        except Exception as e:
            print(f"Assistant (error): {e}\n")
            history.pop()  # rollback failed turn
            continue

        history.append({"role": "assistant", "content": reply})
        print(f"Assistant: {reply}\n")


# Interactive mode: uncomment ONE line below, run ONLY this cell (not "Run all"), then type in the prompt when you see "You:"
# run_health_chat_loop(provider="openai")
# run_health_chat_loop(provider="huggingface")

### One-shot test (no `input` — use this if interactive mode is confusing)

Run the cell below. Edit the question string to try different prompts. Uses **Hugging Face** by default; change `provider` to `"openai"` if you use OpenAI instead.

In [14]:
def ask_once(question: str, provider: Provider = "huggingface") -> str:
    """Send a single user message and print the assistant reply (no input() needed)."""
    risky, disclaimer = safety_filter(question)
    if risky:
        print(f"[Safety] {disclaimer}\n")
    history = [{"role": "user", "content": question}]
    reply = complete_chat(history, provider=provider)
    print(f"You: {question}")
    print(f"Assistant: {reply}\n")
    return reply


# Change the string below, then run this cell
ask_once("What causes a sore throat?", provider="huggingface")
# ask_once("Is paracetamol safe for children?", provider="huggingface")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

You: What causes a sore throat?
Assistant: A sore throat can be caused by a variety of factors, including:

1. Viral infections: Common culprits include the common cold, flu, and tonsillitis. These infections can cause inflammation of the throat tissue, leading to pain and discomfort.

2. Bacterial infections: Bacteria like strep throat, strep tonsillitis, and tonsillitis can cause a sore throat. These infections can be treated with antibiotics.

3. Allergies: Allergic reactions to pollen, dust, or mold can cause a sore throat.

4. Postnasal drip: This is a condition where mucus drips from the nose down the back of the throat, leading to a sore throat.

5. Head trauma: A severe head injury can cause a sore throat as a side effect of the injury.

6. Medication side effects: Some medications can cause a sore throat, such as certain antibiotics, antihistamines, and painkillers.

7. Postnasal drip: This is a condition where mucus drips from the nose down the back of the throat, leading to 

"A sore throat can be caused by a variety of factors, including:\n\n1. Viral infections: Common culprits include the common cold, flu, and tonsillitis. These infections can cause inflammation of the throat tissue, leading to pain and discomfort.\n\n2. Bacterial infections: Bacteria like strep throat, strep tonsillitis, and tonsillitis can cause a sore throat. These infections can be treated with antibiotics.\n\n3. Allergies: Allergic reactions to pollen, dust, or mold can cause a sore throat.\n\n4. Postnasal drip: This is a condition where mucus drips from the nose down the back of the throat, leading to a sore throat.\n\n5. Head trauma: A severe head injury can cause a sore throat as a side effect of the injury.\n\n6. Medication side effects: Some medications can cause a sore throat, such as certain antibiotics, antihistamines, and painkillers.\n\n7. Postnasal drip: This is a condition where mucus drips from the nose down the back of the throat, leading to a sore throat.\n\n8. Cancer:

## 5. Quick test (no API call)

Verify `safety_filter` behavior without spending API credits.

In [15]:
# Examples from task brief
samples = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "I have chest pain and can't breathe",
]
for s in samples:
    hit, disc = safety_filter(s)
    print(f"Q: {s}")
    print(f"  safety_triggered={hit}")
    if disc:
        print(f"  disclaimer: {disc}")
    print()

Q: What causes a sore throat?
  safety_triggered=False

Q: Is paracetamol safe for children?
  safety_triggered=False

Q: I have chest pain and can't breathe
  safety_triggered=True
  disclaimer: I am an AI, not a doctor. If you are experiencing a medical emergency, please call your local emergency services immediately.

